This page is dedicated to utilize BlackJAX package to experiment MCMC sampling process with pathfinder initialization idea from Zhang et al.'s *Pathfinder: Parallel quasi-Newton variational inference*

**To make it run on GPUs**

In [ ]:
!pip uninstall -yq jax jaxlib jax-cuda12-plugin jax-cuda12-pjrt tensorflow-probability
# heads out since jax might drop support for cuda 12, currently (as of Aug 10, 2026), JAX has issues with CUDA 13
# see this post: https://github.com/jax-ml/jax/issues/37923
!pip install -Uq "jax[cuda12]" tfp-nightly blackjax inference_gym optax

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 141.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 124.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 390.9/390.9 kB 39.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 124.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.2/8.2 MB 127.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.8/175.8 MB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.3/87.3 MB 29.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 117.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires tensorflow-probability>=0.13.0, which is not installed.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.5.2 which is incompatible.


Remember to restart Runtime before proceeding

In [ ]:
# run those checks if package compatibility is in trouble

# import jax
# import tensorflow_probability as tfp
# import jaxlib

# print("jaxlib:", jaxlib.__version__)
# print("TFP:", tfp.__version__)

# !pip show jax
# !pip show jaxlib
# !pip show blackjax

# import jax.numpy as jnp

# import blackjax

# import tensorflow_probability.substrates.jax as tfp
# import inference_gym.using_jax as gym

# print("JAX:", jax.__version__)
# print("BlackJAX:", blackjax.__version__)
# print("TFP:", tfp.__version__)
# print("ArviZ:", avs.__version__)
# print("Inference Gym imported successfully!")

**Necessary Packages and Other Settings**

In [ ]:
# GPU set up to accelerate performance
import os
# in case jax eats up my GPU RAM
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ['XLA_FLAGS'] = (
    '--xla_gpu_triton_gemm_any=True '
    '--xla_gpu_enable_latency_hiding_scheduler=true '
)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import jax
import jax.numpy as jnp
from jax import random, jit, vmap, lax
import tensorflow_probability.substrates.jax as tfp
tfd = tfp.distributions
import inference_gym.using_jax as gym
import jaxlib
import blackjax

from blackjax.adaptation.base import get_filter_adapt_info_fn
import optax

# import arviz as az
# import arviz_stats as avs

import warnings
warnings.filterwarnings('ignore')

import psutil

import gc

from google.colab import drive
from matplotlib.lines import Line2D

process = psutil.Process(os.getpid())

def mem(msg):
    print(f"{msg}: {process.memory_info().rss / 1024**2:.1f} MB")

# verification to make sure this is on a GPU
print(jax.devices())
print(jax.default_backend())

[CudaDevice(id=0)]
gpu


In [ ]:
drive.mount('/content/drive')
utility_link = '/content/drive/MyDrive/Colab Notebooks/2026_Summer_MCMC/BJAX_files/PathFinderUtil.py'
with open(utility_link) as f: exec(f.read())

Mounted at /content/drive


In [ ]:
max_warmup = 1000
warmup_window = 100

window_array = np.append(np.repeat(10, 10),
                      np.repeat(warmup_window, max_warmup // warmup_window - 1))

warmup_length = np.repeat(10, len(window_array))
for i in range(len(warmup_length) - 1):
    warmup_length[i + 1] = warmup_length[i] + window_array[i + 1]

# Transition kernel for short regime
repitition = 10
num_chains_short = 2048
num_super_chains = 16

In [ ]:
# quantiles for chi squared with df = 1
chi_up = 3.841459 # 95th quantile for chi squared with df = 1
chi_lo = 0.00393214  # 05th quantile for chi squared with df = 1
tau = 1e-4
M = num_chains_short // num_super_chains
nRhat_lower = np.sqrt(1 + 1 / M + tau)
eps_lower = nRhat_lower - 1
bound = [chi_lo / num_chains_short, chi_up / num_chains_short]
threshold = eps_lower

**Rosenbrock Example**

In [ ]:
target = gym.targets.VectorModel(
    gym.targets.Banana(),
    flatten_sample_transformations=True
)

num_dimensions = target.event_shape[0]

# print("Target:", type(target))
# print("Dimensions:", num_dimensions)
# print("Event shape:", target.event_shape)
# Get some estimates of the mean and variance.
try:
  mean_est = target.sample_transformations['identity'].ground_truth_mean
except:
  print('no ground truth mean')
  mean_est = (result.all_states[num_warmup:, :]).mean(0).mean(0)
try:
  var_est = target.sample_transformations['identity'].ground_truth_standard_deviation**2
except:
  print('no ground truth std dev')
  var_est = ((result.all_states[num_warmup:, :]**2).mean(0).mean(0) -
             mean_est**2)
mean_benchmark = mean_est
var_benchmark = var_est

In [ ]:
def target_log_prob_fn(x):
    y = target.default_event_space_bijector(x)
    fldj = target.default_event_space_bijector.forward_log_det_jacobian(x)
    return target.unnormalized_log_prob(y) + fldj
offset = 2.0
init_step_size = 1.
# def initialize(shape, key):
#     return (10 * random.normal(key, shape+(num_dimensions,))+ offset)
# used to be
# initial_state = initialize_fn((num_sub_chains,),key=ranKey)
def initialize(shape, key):
    return (10 * random.normal(key, shape)+ offset)

# make sure to check initialization before running:
# initialize((num_dimensions,), random.PRNGKey(0))
# to make sure it only initialize one position.

In [ ]:
#simulation part:
Rosen_MSE_p_list = []
Rosen_RHat_p_list = []
Rosen_state_list_p = []

In [ ]:
base_key = random.PRNGKey(0)
keys = random.split(base_key, repitition)
for length in warmup_length:
  mem(f"Simulation Start")
  simulation(length,num_chains_short, num_super_chains,
             initialize, keys,
             target_log_prob_fn,init_step_size,
             repitition, Rosen_RHat_p_list,Rosen_MSE_p_list,
             mean_benchmark,var_benchmark, num_dimensions)

Simulation Start: 1470.3 MB
Warmup Length: 10; mean of MSE is: 0.5060300230979919
Simulation Start: 2588.3 MB
Warmup Length: 20; mean of MSE is: 0.4436664581298828
Simulation Start: 2597.3 MB
Warmup Length: 30; mean of MSE is: 0.40434855222702026
Simulation Start: 2607.7 MB
Warmup Length: 40; mean of MSE is: 0.3584152162075043
Simulation Start: 2610.8 MB
Warmup Length: 50; mean of MSE is: 0.321159303188324
Simulation Start: 2612.8 MB
Warmup Length: 60; mean of MSE is: 0.3011251389980316
Simulation Start: 2618.0 MB
Warmup Length: 70; mean of MSE is: 0.2768482565879822
Simulation Start: 2618.8 MB
Warmup Length: 80; mean of MSE is: 0.24975451827049255
Simulation Start: 2619.1 MB
Warmup Length: 90; mean of MSE is: 0.2328767478466034
Simulation Start: 2619.6 MB
Warmup Length: 100; mean of MSE is: 0.21009314060211182
Simulation Start: 2620.0 MB
Warmup Length: 200; mean of MSE is: 0.12495732307434082
Simulation Start: 2620.2 MB
Warmup Length: 300; mean of MSE is: 0.08861517906188965
Simulatio

In [ ]:
RHat_p_df = pd.DataFrame(Rosen_RHat_p_list)
MSE_p_df = pd.DataFrame(Rosen_MSE_p_list)

In [ ]:
MSE_p_df.to_pickle(
    "/content/drive/MyDrive/Colab Notebooks/2026_Summer_MCMC/BlackJAX_PF_pkl_files/Rosen_MSE.pkl"
)

RHat_p_df.to_pickle(
    "/content/drive/MyDrive/Colab Notebooks/2026_Summer_MCMC/BlackJAX_PF_pkl_files/Rosen_R_Hat.pkl"
)

**Bimodal Example**

In [ ]:
num_dimensions = 100
offset = 0.0
init_step_size = 1.0

def target_log_prob_fn(x):
    logp1 = (
        jnp.log(0.3)
        - 0.5 * jnp.sum((x + 5.0) ** 2)
        - 0.5 * num_dimensions * jnp.log(2 * jnp.pi)
    )

    logp2 = (
        jnp.log(0.7)
        - 0.5 * jnp.sum((x - 5.0) ** 2)
        - 0.5 * num_dimensions * jnp.log(2 * jnp.pi)
    )

    return jax.scipy.special.logsumexp(
        jnp.array([logp1, logp2])
    )

def initialize (shape, key):
  return 10 * random.normal(key, shape) + offset

# make sure to check initialization before running:
# initialize((num_dimensions,), random.PRNGKey(0))
# to make sure it only initialize one position.

mean_est = jnp.repeat(2, num_dimensions)
var_est = jnp.repeat(22, num_dimensions)
mean_benchmark = mean_est
var_benchmark = var_est

In [ ]:
#simulation part:
Bim_MSE_p_list = []
Bim_RHat_p_list = []
Bim_state_list_p = []

In [ ]:
base_key = random.PRNGKey(0)
keys = random.split(base_key, repitition)
for length in warmup_length:
  mem(f"Simulation Start")
  simulation(length,num_chains_short, num_super_chains,
             initialize, keys,
             target_log_prob_fn,init_step_size,
             repitition, Bim_RHat_p_list,Bim_MSE_p_list,
             mean_benchmark,var_benchmark, num_dimensions)

Simulation Start: 1461.5 MB
Warmup Length: 10; mean of MSE is: 1.3182077407836914
Simulation Start: 2642.3 MB
Warmup Length: 20; mean of MSE is: 1.3183066844940186
Simulation Start: 2650.1 MB
Warmup Length: 30; mean of MSE is: 1.3185744285583496
Simulation Start: 2660.6 MB
Warmup Length: 40; mean of MSE is: 1.318537950515747
Simulation Start: 2669.2 MB
Warmup Length: 50; mean of MSE is: 1.3184887170791626
Simulation Start: 2679.4 MB
Warmup Length: 60; mean of MSE is: 1.3187079429626465
Simulation Start: 2687.5 MB
Warmup Length: 70; mean of MSE is: 1.319044828414917
Simulation Start: 2695.6 MB
Warmup Length: 80; mean of MSE is: 1.3182073831558228
Simulation Start: 2704.3 MB
Warmup Length: 90; mean of MSE is: 1.3181158304214478
Simulation Start: 2712.6 MB
Warmup Length: 100; mean of MSE is: 1.318312168121338
Simulation Start: 2715.4 MB
Warmup Length: 200; mean of MSE is: 1.3188722133636475
Simulation Start: 2723.8 MB
Warmup Length: 300; mean of MSE is: 1.3183848857879639
Simulation Start

In [ ]:
RHat_p_df = pd.DataFrame(Bim_RHat_p_list)
MSE_p_df = pd.DataFrame(Bim_MSE_p_list)
MSE_p_df.to_pickle(
    "/content/drive/MyDrive/Colab Notebooks/2026_Summer_MCMC/BlackJAX_PF_pkl_files/Bim_MSE.pkl"
)

RHat_p_df.to_pickle(
    "/content/drive/MyDrive/Colab Notebooks/2026_Summer_MCMC/BlackJAX_PF_pkl_files/Bim_R_Hat.pkl"
)

**Eight School Example**

In [ ]:
# NOTE: inference gym stores the centered parameterization
target_raw = gym.targets.EightSchools()  # store raw to examine doc.
target = gym.targets.VectorModel(target_raw,
                                  flatten_sample_transformations = True)
num_dimensions = target.event_shape[0]
init_step_size = 1.

def initialize (shape, key):
    prior_scale = jnp.append(jnp.array([10., 1.]), jnp.repeat(1., 8))
    prior_offset = jnp.append(jnp.array([0., 5.]), jnp.repeat(0., 8))
    return prior_scale * random.normal(key, shape) + prior_offset

num_schools = 8
y = np.array([28, 8, -3, 7, -1, 1, 18, 12], dtype = np.float32)
sigma = np.array([15, 10, 16, 11, 9, 11, 10, 18], dtype = np.float32)

# NOTE: the reinterpreted batch dimension specifies the dimension of
# each indepdent variable, here the school.
model = tfd.JointDistributionSequential([
    tfd.Normal(loc = 0., scale = 10., name = "mu"),
    tfd.Normal(loc = 5., scale = 1., name = "log_tau"),
    tfd.Independent(tfd.Normal(loc = jnp.zeros(num_schools),
                               scale = jnp.ones(num_schools),
                               name = "eta"),
                    reinterpreted_batch_ndims = 1),
    lambda eta, log_tau, mu: (
        tfd.Independent(tfd.Normal(loc = (mu[..., jnp.newaxis] +
                                        jnp.exp(log_tau[..., jnp.newaxis]) *
                                        eta),
                                   scale = sigma),
                        name = "y",
                        reinterpreted_batch_ndims = 1))
  ])

# minor change from the TFP code
# def target_log_prob_fn(x):
#   mu = x[:, 0]
#   log_tau = x[:, 1]
#   eta = x[:, 2:10]
#   return model.log_prob((mu, log_tau, eta, y))
def target_log_prob_fn(x):
  mu = x[0]
  log_tau = x[1]
  eta = x[2:10]
  return model.log_prob((mu, log_tau, eta, y))

In [ ]:
# Use results from running 128 chains with 1000 + 5000 iterations each,
# for non-centered parameterization.
mean_est = np.array([5.8006573 ,  2.4502006 ,  0.6532423 ,  0.09639207,
             -0.23725411,  0.04723661, -0.33556408, -0.19666635,
              0.5390533 ,  0.14633301])

var_est = np.array([29.60382   ,  0.26338503,  0.6383733 ,  0.4928926 ,
              0.65307987,  0.52441144,  0.46658015,  0.5248887 ,
              0.49544162,  0.690975])
mean_benchmark = mean_est
var_benchmark = var_est

In [ ]:
#simulation part:
School_MSE_p_list = []
School_RHat_p_list = []
School_state_list_p = []

In [ ]:
base_key = random.PRNGKey(0)
keys = random.split(base_key, repitition)
for length in warmup_length:
  mem(f"Simulation Start")
  simulation(length,num_chains_short, num_super_chains,
             initialize, keys,
             target_log_prob_fn,init_step_size,
             repitition, School_RHat_p_list,School_MSE_p_list,
             mean_benchmark,var_benchmark, num_dimensions)

Simulation Start: 1477.8 MB
Warmup Length: 10; mean of MSE is: 1.442039132118225
Simulation Start: 2579.7 MB
Warmup Length: 20; mean of MSE is: 1.4177162647247314
Simulation Start: 2590.6 MB
Warmup Length: 30; mean of MSE is: 1.4016910791397095
Simulation Start: 2592.0 MB
Warmup Length: 40; mean of MSE is: 1.3884327411651611
Simulation Start: 2592.7 MB
Warmup Length: 50; mean of MSE is: 1.3781973123550415
Simulation Start: 2595.1 MB
Warmup Length: 60; mean of MSE is: 1.3664700984954834
Simulation Start: 2597.6 MB
Warmup Length: 70; mean of MSE is: 1.3576691150665283
Simulation Start: 2598.0 MB
Warmup Length: 80; mean of MSE is: 1.348921775817871
Simulation Start: 2598.7 MB
Warmup Length: 90; mean of MSE is: 1.3400413990020752
Simulation Start: 2599.2 MB
Warmup Length: 100; mean of MSE is: 1.3343772888183594
Simulation Start: 2599.8 MB
Warmup Length: 200; mean of MSE is: 1.2500132322311401
Simulation Start: 2600.1 MB
Warmup Length: 300; mean of MSE is: 1.160382628440857
Simulation Start

In [ ]:
RHat_p_df = pd.DataFrame(School_RHat_p_list)
MSE_p_df = pd.DataFrame(School_MSE_p_list)
MSE_p_df.to_pickle(
    "/content/drive/MyDrive/Colab Notebooks/2026_Summer_MCMC/BlackJAX_PF_pkl_files/School_MSE.pkl"
)

RHat_p_df.to_pickle(
    "/content/drive/MyDrive/Colab Notebooks/2026_Summer_MCMC/BlackJAX_PF_pkl_files/School_R_Hat.pkl"
)

**Item Response Theory Example**

In [ ]:
target = gym.targets.VectorModel(gym.targets.SyntheticItemResponseTheory(),
                                 flatten_sample_transformations=True)
num_dimensions = target.event_shape[0]
init_step_size = 1.

def target_log_prob_fn(x):
  """Unnormalized, unconstrained target density.

  This is a thin wrapper that applies the default bijectors so that we can
  ignore any constraints.
  """
  y = target.default_event_space_bijector(x)
  fldj = target.default_event_space_bijector.forward_log_det_jacobian(x)
  return target.unnormalized_log_prob(y) + fldj

offset = 0
def initialize (shape, key):
  return 10 * random.normal(key, shape) + offset

In [ ]:
# Get some estimates of the mean and variance.
try:
  mean_est = target.sample_transformations['identity'].ground_truth_mean
except:
  print('no ground truth mean')
  mean_est = (result.all_states[num_warmup:, :]).mean(0).mean(0)
try:
  var_est = target.sample_transformations['identity'].ground_truth_standard_deviation**2
except:
  print('no ground truth std dev')
  var_est = ((result.all_states[num_warmup:, :]**2).mean(0).mean(0) -
             mean_est**2)

mean_benchmark = mean_est
var_benchmark = var_est

In [ ]:
#simulation part:
IRT_MSE_p_list = []
IRT_RHat_p_list = []
IRT_state_list_p = []

In [ ]:
base_key = random.PRNGKey(0)
keys = random.split(base_key, repitition)
for length in warmup_length:
  mem(f"Simulation Start")
  simulation(length,num_chains_short, num_super_chains,
             initialize, keys,
             target_log_prob_fn,init_step_size,
             repitition, IRT_RHat_p_list,IRT_MSE_p_list,
             mean_benchmark,var_benchmark, num_dimensions)

Simulation Start: 1471.7 MB
Warmup Length: 10; mean of MSE is: 65733.4453125
Simulation Start: 3159.4 MB
Warmup Length: 20; mean of MSE is: 46383.1796875
Simulation Start: 3202.0 MB
Warmup Length: 30; mean of MSE is: 22566.15625
Simulation Start: 3244.1 MB
Warmup Length: 40; mean of MSE is: 14576.4482421875
Simulation Start: 3285.7 MB
Warmup Length: 50; mean of MSE is: 10132.8544921875
Simulation Start: 3326.6 MB
Warmup Length: 60; mean of MSE is: 7378.9228515625
Simulation Start: 3369.0 MB
Warmup Length: 70; mean of MSE is: 5559.93798828125
Simulation Start: 3410.3 MB
Warmup Length: 80; mean of MSE is: 4369.4365234375
Simulation Start: 3451.0 MB
Warmup Length: 90; mean of MSE is: 3586.731201171875
Simulation Start: 3492.1 MB
Warmup Length: 100; mean of MSE is: 3034.61962890625
Simulation Start: 3533.1 MB
Warmup Length: 200; mean of MSE is: 1091.906005859375
Simulation Start: 3574.0 MB
Warmup Length: 300; mean of MSE is: 638.7808227539062
Simulation Start: 3618.1 MB
Warmup Length: 400;

In [ ]:
RHat_p_df = pd.DataFrame(IRT_RHat_p_list)
MSE_p_df = pd.DataFrame(IRT_MSE_p_list)
MSE_p_df.to_pickle(
    "/content/drive/MyDrive/Colab Notebooks/2026_Summer_MCMC/BlackJAX_PF_pkl_files/IRT_MSE.pkl"
)

RHat_p_df.to_pickle(
    "/content/drive/MyDrive/Colab Notebooks/2026_Summer_MCMC/BlackJAX_PF_pkl_files/IRT_R_Hat.pkl"
)